# Scheme Success Model
Condensed version. Part 1 predicts Expected_Leads / Expected_Bookings / Estimated_Conversion_Pct / Scheme_Success_Score together (regression). Part 2 predicts Demand_Level (classification).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               RandomForestClassifier, GradientBoostingClassifier)
from sklearn.metrics import r2_score, accuracy_score, classification_report

df = pd.read_csv("../data/raw/historical_scheme_data.csv")
df["EMI_to_Price_Ratio"] = df["Monthly_EMI_INR"] / df["Price_INR"]
df["AdBudget_per_Sqft"] = df["Advertising_Budget_INR"] / df["Plot_Size_SqFt"]
print(df.shape)


(800, 18)


## Part 1 - Multi-output Regression
Only real "known before you launch it" inputs - Expected_Leads/Bookings/Conversion
are outputs, not inputs, so they don't go in here.

In [ ]:
input_features = ["Location", "Plot_Size_SqFt", "Price_INR", "Monthly_EMI_INR",
                   "Park", "Clubhouse", "Distance_from_Metro_km", "Advertising_Budget_INR",
                   "EMI_to_Price_Ratio", "AdBudget_per_Sqft"]
target_cols = ["Expected_Leads", "Expected_Bookings", "Estimated_Conversion_Pct", "Scheme_Success_Score"]

X = pd.get_dummies(df[input_features], columns=["Location", "Park", "Clubhouse"], drop_first=True)
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Training set: {X_train.shape}, Testing set: {X_test.shape}")
print(f"Training set: {y_train.shape}, Testing set: {y_test.shape}")

Linear Regression: avg R2=0.669


Tuned Random Forest: avg R2=0.641


Tuned Gradient Boosting: avg R2=0.733
best: Tuned Gradient Boosting


In [ ]:
def evaluate_multi(model, name):
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    r2 = r2_score(y_test, pred, multioutput="uniform_average")
    print(f"{name}: avg R2={r2:.3f}")
    return model, r2

lr_model, lr_r2 = evaluate_multi(LinearRegression(), "Linear Regression")

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), {"n_estimators":[100,200], "max_depth":[10,15]}, cv=3, scoring="r2")
rf_grid.fit(X_train_s, y_train)
rf_model, rf_r2 = evaluate_multi(rf_grid.best_estimator_, "Tuned Random Forest")

gb_grid = GridSearchCV(MultiOutputRegressor(GradientBoostingRegressor(random_state=42)),
                        {"estimator__n_estimators":[100,200], "estimator__learning_rate":[0.05,0.1], "estimator__max_depth":[2,3]},
                        cv=3, scoring="r2")
gb_grid.fit(X_train_s, y_train)
gb_model, gb_r2 = evaluate_multi(gb_grid.best_estimator_, "Tuned Gradient Boosting")

best_reg_name, best_reg_model = max(
    [("Linear Regression", lr_model), ("Tuned Random Forest", rf_model), ("Tuned Gradient Boosting", gb_model)],
    key=lambda t: r2_score(y_test, t[1].predict(X_test_s), multioutput="uniform_average"))
print("best:", best_reg_name)

In [3]:
import joblib
joblib.dump(best_reg_model, "../models/scheme_multi_output_model.pkl")
joblib.dump(scaler, "../models/scheme_reg_scaler.pkl")
joblib.dump(list(X.columns), "../models/scheme_reg_columns.pkl")
joblib.dump(target_cols, "../models/scheme_reg_targets.pkl")
print("saved:", best_reg_name)


saved: Tuned Gradient Boosting


## Part 2 - Classification (Demand_Level)
Not using Scheme_Success_Score as a feature - Demand_Level is literally binned from it
(LOW=20-49, MEDIUM=50-74, HIGH=75-98), so that would be a 100% accuracy cheat.
class_weight="balanced" since HIGH only has 25 test rows.

In [4]:
Xc = pd.get_dummies(df[input_features], columns=["Location", "Park", "Clubhouse"], drop_first=True)
yc = df["Demand_Level"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
clf_scaler = MinMaxScaler()
Xc_train_s = clf_scaler.fit_transform(Xc_train)
Xc_test_s = clf_scaler.transform(Xc_test)

def evaluate_clf(model, name):
    model.fit(Xc_train_s, yc_train)
    pred = model.predict(Xc_test_s)
    acc = accuracy_score(yc_test, pred)
    print(f"{name}: accuracy={acc:.3f}")
    return model, acc

lr_clf, lr_acc = evaluate_clf(LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
gb_clf, gb_acc = evaluate_clf(GradientBoostingClassifier(n_estimators=200, random_state=42), "Gradient Boosting")

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, class_weight="balanced"),
                        {"n_estimators":[100,200], "max_depth":[8,15]}, cv=3, scoring="f1_macro")
rf_grid.fit(Xc_train_s, yc_train)
rf_clf, rf_acc = evaluate_clf(rf_grid.best_estimator_, "Tuned Random Forest")

best_clf_name, best_clf_model = max(
    [("Logistic Regression", lr_clf), ("Gradient Boosting", gb_clf), ("Tuned Random Forest", rf_clf)],
    key=lambda t: accuracy_score(yc_test, t[1].predict(Xc_test_s)))
print("best:", best_clf_name)
print(classification_report(yc_test, best_clf_model.predict(Xc_test_s)))


Logistic Regression: accuracy=0.662


Gradient Boosting: accuracy=0.750


Tuned Random Forest: accuracy=0.787
best: Tuned Random Forest
              precision    recall  f1-score   support

        HIGH       0.83      0.76      0.79        25
         LOW       0.79      0.75      0.77        56
      MEDIUM       0.77      0.82      0.80        79

    accuracy                           0.79       160
   macro avg       0.80      0.78      0.79       160
weighted avg       0.79      0.79      0.79       160



In [5]:
joblib.dump(best_clf_model, "../models/scheme_demand_classifier.pkl")
joblib.dump(clf_scaler, "../models/scheme_demand_scaler.pkl")
joblib.dump(list(Xc.columns), "../models/scheme_demand_columns.pkl")
print("saved:", best_clf_name)


saved: Tuned Random Forest
